# 2.2 HPC Sites

HPC sites stage the same project/job artifacts to a remote work directory, submit through a scheduler, then fetch logs and outputs. The concrete public implementation shown here is `Stampede3Site`, which supplies Stampede3/TACC defaults on top of the generic SLURM machinery.

By the end, you should understand which parts of an HPC run are ordinary FrequenSolve jobs and which parts belong to remote paths, scheduler settings, modules, and solver installation.


## Configuration Checklist

HPC sites use the same project and job objects, but staging, scheduler settings, and remote filesystem conventions become part of the site configuration. The Python objects stay deliberately consistent across sites: a `Project` owns simulations and jobs, a `Job` describes work, and a `Site` decides where that work runs.

| Concern | AWS site | HPC site | Local site |
| --- | --- | --- | --- |
| Solver availability | Managed in the cloud environment. | Must be installed on the target cluster. | Must be installed on this workstation. |
| Staging | Uploads project/job artifacts to cloud storage. | Copies or syncs artifacts to a remote work directory. | Writes artifacts directly under the project path. |
| Execution | Submitted through the cloud service. | Submitted through the scheduler, usually SLURM. | Runs a local worker process. |
| Results | Fetched from cloud storage through the result API. | Fetched from the remote work directory. | Read directly from local result directories. |
| Best first use | Production and normal user workflows. | Institution-managed clusters with solver access. | API development, smoke tests, and small examples. |

The run cells below are intentionally strict. If SSH access, remote paths, SLURM settings, modules, or solver installation are wrong, the cell should fail with enough site/job context to inspect logs rather than hiding the problem behind a broad exception handler.


## Site Mental Model

A site changes where a completed job runs; it should not change how the simulation is authored or how results are read. Keep the project, simulation, acquisition, jobs, trace reads, and ParaView output requests ordinary. Let the site object handle authentication, staging, scheduling, storage, polling, and fetching.

That separation is what makes it possible to prototype locally, run production jobs in the cloud or on HPC, and keep the analysis cells nearly identical.


## Imports And Shared Job Builder

The helper creates one simulation and two jobs: a time-domain trace job and a single-frequency ParaView QC job. That pair is repeated across all site tutorials so the only moving part is the `Site` object.

The builder returns project-owned jobs rather than loose JSON. Calling `site.submit(job)` serializes the simulation, job, acquisition, mesh, outputs, and units into the project structure before staging or execution. That is the core site contract users should remember: author locally, submit through the selected site, fetch through the returned result handle.


In [ ]:

import numpy as np
import frequensolve as fs

u = fs.ureg


In [ ]:
def build_acoustic_tutorial_jobs(project_path, *, simulation_name, trace_job_name, qc_job_name, f_max=25.0):
    project = fs.Project(
        name="project",
        pretty_name=simulation_name,
        path=project_path,
        log_level="INFO",
        log_to_console=True,
    )
    sim = project.new_simulation(
        name=simulation_name,
        physics="acoustic",
        dimension=2,
        units={"length": "km", "velocity": "km/s", "density": "g/cm^3"},
    )

    model = fs.LayeredModel(name="model", dimension=2, x_limits=[0.0, 1.0])
    model.add_surface(name="top", depth=0.0 * u.km)
    model.add_layer(name="water", properties={"Vp": 1.5 * u.km / u.s, "Rho": 1.0 * u.g / u.cm**3})
    model.add_surface(name="interface", depth=0.22 * u.km)
    model.add_layer(name="basement", properties={"Vp": 2.4 * u.km / u.s, "Rho": 2.2 * u.g / u.cm**3})
    model.add_surface(name="bottom", depth=0.5 * u.km)
    sim += model

    sim += model.hex_mesh_generator([8, 4])
    sim.mesh.set_adapt(elems_per_wave=2.0, order=4, f_low=5.0, f_high=f_max)
    sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
    sim += fs.BoundaryCondition(conditions=["pml"], boundaries=["x_min", "x_max", "z_max"], pml_wavelengths=0.75)

    acq = fs.Acquisition(max_batch=2)
    acq.add_source_group(kind="scalar", coords=[[0.35, 0.05], [0.65, 0.05]])
    hydrophone = fs.ReceiverNode(name="hydrophone")
    hydrophone.add_component(name="p", field="pressure")
    acq.add_receiver_group(name="surface", device=hydrophone, coords=[[x, 0.04] for x in np.linspace(0.1, 0.9, 61)])
    sim += acq
    sim += fs.Discretization()
    sim += fs.SolverConfig(tolerance=1.0e-4, grids=3)

    trace_job = fs.TimeDomainJob(
        name=trace_job_name,
        simulation=sim,
        f_min=0.0,
        f_max=f_max,
        T_max=0.9,
    )
    qc_job = fs.FrequencyDomainJob(
        name=qc_job_name,
        simulation=sim,
        f_list=[12.0],
        outputs=[
            fs.ParaviewOutput(
                name="pv_qc",
                fields=["pressure"],
                properties=["vp", "rho", "Subdomain"],
                show_pml=True,
                upscale=0,
                order=1,
            )
        ],
    )
    return project, sim, trace_job, qc_job


## Author The Jobs And SLURM Settings

The time-domain job writes traces; the single-frequency QC job writes ParaView output. The SLURM settings describe the remote allocation used by both jobs. Adjust queue, duration, and process counts to match the target cluster policy.

The QC job uses `upscale=0` to keep staged visualization artifacts compact. Increase it for smoother post-run images after the site workflow is confirmed.


In [ ]:
project, sim, trace_job, qc_job = build_acoustic_tutorial_jobs(
    "./scratch/tutorials/hpc_site",
    simulation_name="hpc_site_acoustic",
    trace_job_name="time_hpc_site",
    qc_job_name="freq_hpc_site_qc",
)
project.save()

run_config = fs.SlurmRunConfig(
    queue="skx-dev",
    nodes=1,
    duration="00:30:00",
    procs_per_node=4,
    procs_per_task=1,
    poll_interval=10,
)
{
    "trace_job": trace_job.to_fs(),
    "qc_job": qc_job.to_fs(),
    "run_config": run_config,
}


## Configure And Submit To Stampede3

`Stampede3Site` stages the project into the configured remote path, writes SLURM launch scripts, polls status, and downloads the result bundle. Jobs can only run on HPC systems where the fast solver and runtime environment are installed.


In [ ]:
site = fs.Stampede3Site(
    rel_path="scratch/frequensolve_tutorials/hpc_site",
    run_config=run_config,
    verbose=True,
)
trace_result = site.submit(trace_job).wait()
qc_result = site.submit(qc_job).wait()
{
    "trace_status": trace_result.status,
    "qc_status": qc_result.status,
}


## Fetch Remote Results

The result handle exposes the same API after an HPC run. Logs are the first thing to fetch when a job fails; traces and ParaView files are the first things to inspect when it succeeds.


In [ ]:
logs = trace_result.logs()
qc_outputs = qc_result.output_files(existing=True)
traces = trace_result.traces(upscale=4)
{
    "trace_successful": trace_result.successful,
    "qc_successful": qc_result.successful,
    "logs": str(logs),
    "qc_output_count": len(qc_outputs),
    "trace_files": traces.files,
    "frequency_summary": trace_job.frequency_summary(),
}


## Plot A Retrieved HPC Trace Gather


In [ ]:
traces = trace_result.traces(upscale=4)
wavelet = fs.RickerWavelet(f=12.0)
group = traces.groups[0]
component = traces.components(group)[0]
source = traces.sources(group)[0]
gather = traces.td(group, component, source, wavelet, upscale=4, T_max=0.9)
fs.plot_gather(
    gather,
    A=2.0 * np.nanstd(np.real(gather.values)),
    cmap="gray",
    figsize=(9, 4),
    title=f"{trace_job.name}: {group}/{component}/source {source}",
)


## Result Review Checklist

The HPC-specific part of this notebook is staging plus scheduler configuration. The job objects should remain ordinary FrequenSolve jobs; the site decides how those jobs become SLURM scripts, remote work directories, and fetched artifacts.

| Artifact | What to confirm |
| --- | --- |
| `SlurmRunConfig` | Queue, node count, task layout, duration, and polling interval match cluster policy. |
| Remote path | `rel_path` points to a writable project area that will not collide with unrelated runs. |
| Logs | Scheduler logs and solver logs are both reachable after `wait()`. |
| Output API | Traces and ParaView files are consumed with the same methods as local/AWS results. |

When debugging HPC submissions, inspect the generated job script and scheduler log before changing the simulation. Most failures at this layer are environment, module, path, or allocation issues rather than model-definition issues.
